# Smartphone Usage and Addiction Prediction: Model Training

## 1. Project Introduction

This project uses daily behavior and wellbeing information to identify people at risk of smartphone addiction. We predict **`addicted_label`**. This is a **classification** problem because the answer is one of a fixed set of labels. A reliable prediction can support early, supportive digital-wellbeing interventions.

This notebook follows a leakage-safe workflow and saves preprocessing and prediction logic together for a future FastAPI service.

## 2. Import Libraries

Pandas and NumPy handle tabular data, Matplotlib creates focused charts, Scikit-learn supplies splitting, preprocessing, pipelines, models, and metrics, and Joblib persists the final pipeline.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, precision_score,
                             recall_score, roc_auc_score, RocCurveDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

## 3. Load the Dataset

The CSV is addressed relative to the notebook, not with a machine-specific absolute path. Run this notebook with its project folder as the working directory.

In [ ]:
DATA_PATH = Path("Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## 4. Understand the Dataset

These checks reveal its dimensions, column names and types, descriptive statistics, missing values, and exact duplicate rows. The target is **`addicted_label`**. Usage durations, notifications, app opens, age, and sleep are numeric. Gender, stress, and work/academic impact are categorical. `transaction_id` and `user_id` are identifiers. `addiction_level` is excluded because it is a post-assessment summary closely encoding the binary target and would leak the answer.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isnull().sum().to_frame("missing"))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
target_column = "addicted_label"
identifier_columns = ['transaction_id', 'user_id']
print("Target distribution:")
display(df[target_column].value_counts(dropna=False).to_frame("count"))
print("Identifier columns to remove:", identifier_columns)

## 5. Exploratory Data Analysis

The first plot checks class balance. The second focuses on a useful numeric relationship: **daily screen time by addiction label**. We keep the EDA intentionally small so each chart answers a question.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[target_column].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set(title="Target distribution", xlabel=target_column, ylabel="Count")
df.boxplot(column="daily_screen_time_hours", by=target_column, ax=axes[1]); axes[1].set(title="Screen time by addiction label", xlabel="Addicted label", ylabel="Hours/day"); plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Data Cleaning

Numeric CSV fields are converted explicitly. `addiction_level` is removed later as target leakage: a future risk prediction should be based on behavior available before the addiction assessment, not another version of its result. Exact duplicates are removed before splitting. Identifier fields (transaction_id, user_id) describe record identity rather than a reusable pattern and are excluded. Missing feature values are deliberately handled **inside the pipeline**, which prevents information from the test set influencing training.

In [ ]:
clean_df = df.drop_duplicates().copy()
numeric_columns = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
                   "work_study_hours", "sleep_hours", "notifications_per_day",
                   "app_opens_per_day", "weekend_screen_time"]
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
clean_df = clean_df.drop(columns=identifier_columns)
print("Cleaned shape:", clean_df.shape)

## 7. Prepare Features and Target

`X` contains the input features; `y` contains the answer the model learns to predict. Separating them prevents the target from accidentally becoming an input.

In [ ]:
X = clean_df.drop(columns=[target_column] + ['addiction_level'])
y = clean_df[target_column]
print("X shape:", X.shape, "| y shape:", y.shape)
print("Feature columns:", X.columns.tolist())

## 8. Train/Test Split

The training set teaches the models; the untouched test set estimates performance on unseen records. Evaluating on training data would be overly optimistic. Stratification preserves class proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 9. Preprocessing

Columns are detected from the training data only. Numeric values receive median imputation and standardization. Categorical values receive most-frequent imputation and one-hot encoding; unknown future categories are safely ignored. `ColumnTransformer` applies each branch, and each model is attached in a complete `Pipeline`.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

## 10. Train Multiple Models

Logistic Regression is a strong, interpretable baseline. Random Forest captures nonlinear relationships and interactions. Both are appropriate classifiers, and comparing them is more informative than choosing a complicated model without evidence.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
}
pipelines = {}
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    pipelines[name] = pipeline
print("Trained:", list(pipelines))

## 11. Evaluate Models

Accuracy measures all correct predictions. Macro precision, recall, and F1 give each class equal importance. For binary problems, ROC-AUC measures ranking across probability thresholds. Confusion matrices and classification reports show which classes cause errors.

In [ ]:
results = []
for name, pipeline in pipelines.items():
    predictions = pipeline.predict(X_test)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (macro)": precision_score(y_test, predictions, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, predictions, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, predictions, average="macro", zero_division=0),
    }
    if len(pipeline.classes_) == 2:
        row["ROC-AUC"] = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
    results.append(row)
results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 (macro)", ascending=False)
display(results_df.round(4))

In [ ]:
for name, pipeline in pipelines.items():
    predictions = pipeline.predict(X_test)
    print(f"\n{name}\n", classification_report(y_test, predictions, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Blues")
    plt.title(f"{name} confusion matrix")
    plt.show()
    if len(pipeline.classes_) == 2:
        RocCurveDisplay.from_estimator(pipeline, X_test, y_test)
        plt.title(f"{name} ROC curve")
        plt.show()

## 12. Select and Save the Final Model

Macro F1 is the selection metric because it balances precision and recall across classes, including minority classes. Selecting from the held-out comparison keeps the decision transparent. The saved object is the **entire fitted pipeline**, not only its classifier.

In [ ]:
best_model_name = results_df.index[0]
final_pipeline = pipelines[best_model_name]
print("Selected model:", best_model_name)
print(results_df.loc[best_model_name].round(4))

MODEL_PATH = Path("models/smartphone_addiction_model.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, MODEL_PATH)
print("Saved complete pipeline to:", MODEL_PATH)

## 13. Test With New Data and Verify the Saved Model

This observation has the same raw feature schema a future API request must supply. We reload the on-disk object before predicting, proving that persisted preprocessing and model steps work together. Probabilities express model confidence, not certainty.

In [ ]:
new_observation = pd.DataFrame([{'age': 24, 'gender': 'Female', 'daily_screen_time_hours': 8.2, 'social_media_hours': 4.1, 'gaming_hours': 1.3, 'work_study_hours': 5.0, 'sleep_hours': 5.8, 'notifications_per_day': 210, 'app_opens_per_day': 130, 'weekend_screen_time': 9.0, 'stress_level': 'High', 'academic_work_impact': 'Yes'}])[X.columns]
loaded_pipeline = joblib.load(MODEL_PATH)
new_prediction = loaded_pipeline.predict(new_observation)[0]
new_probabilities = loaded_pipeline.predict_proba(new_observation)[0]
print("Reloaded object type:", type(loaded_pipeline).__name__)
print("Prediction:", new_prediction)
print("Class probabilities:", dict(zip(loaded_pipeline.classes_, new_probabilities.round(4))))
assert len(loaded_pipeline.steps) == 2
assert loaded_pipeline.predict(new_observation).shape == (1,)
print("Verification passed: the saved complete pipeline reloaded and predicted successfully.")

## Conclusion

The dataset has been explored, cleaned without test leakage, used to compare two suitable classifiers, and converted into a reloadable deployment artifact. The next stage—not implemented here—is to load this pipeline inside FastAPI.